# Notebook 00: Example 1, what makes a point anomalous

**Class 03_01** (anomaly detection I). The running example of the lecture, rebuilt so that every number on the
slides can be checked: one host's sessions described by two standardised features, $x_1$ = log bytes sent and
$x_2$ = log bytes received. They are strongly correlated ($\rho = 0.9$): a session that sends more also receives more.

* **Normal model:** $\mathbf{x}\sim\mathcal{N}(\boldsymbol\mu,C)$ with $\boldsymbol\mu=(0,0)$,
  $C=\begin{pmatrix}1&0.9\\0.9&1\end{pmatrix}$.
* **Point A** $=(2,2)$: a big but *consistent* session. **Point B** $=(1,-1)$: average sizes, but it sends a lot and
  receives little (the shape of an exfiltration).

Sections: A. z-score vs Mahalanobis; B. thresholds and the base rate; C. contamination and robust estimates (MCD);
D. distance, density and isolation on the same data (kNN, LOF, Isolation Forest).

In [ ]:
import sys

import numpy as np
import polars as pl
from matplotlib import pyplot as plt
from scipy.stats import chi2

sys.path.insert(0, ".")
import anomlib as an

plt.rcParams["figure.dpi"] = 100
an.table_config()
X = an.example1(500, seed=0)
A, B = an.EX1_POINTS["A"], an.EX1_POINTS["B"]
print("sample mean", X.mean(0).round(3), " sample correlation", np.corrcoef(X.T)[0, 1].round(3))

## A. z-score vs Mahalanobis distance

The **z-score** looks at one feature at a time: $z_j=(x_j-\mu_j)/\sigma_j$, and a point is anomalous when
$\max_j|z_j|$ is large. The **Mahalanobis distance** uses the covariance:
$d_M(\mathbf{x})^2=(\mathbf{x}-\boldsymbol\mu)^\top C^{-1}(\mathbf{x}-\boldsymbol\mu)$.
By hand with the true parameters: $C^{-1}=\frac{1}{0.19}\begin{pmatrix}1&-0.9\\-0.9&1\end{pmatrix}$, so
$d_M(A)^2=(4+4-7.2)/0.19=4.21$ and $d_M(B)^2=(1+1+1.8)/0.19=20.0$.

In [ ]:
Sinv = np.linalg.inv(an.EX1_COV)
rows = []
for name, p in [("A", A), ("B", B)]:
    d2 = float(p @ Sinv @ p)
    rows.append({"point": name, "max |z|": float(np.abs(p).max()), "Euclidean": float(np.linalg.norm(p)),
                 "Mahalanobis^2": round(d2, 2), "Mahalanobis": round(np.sqrt(d2), 2),
                 "P(chi2_2 > d^2)": float(chi2.sf(d2, 2))})  # fmt: skip
print(pl.DataFrame(rows))
print(f"det C = {np.linalg.det(an.EX1_COV):.2f}; chi2 tail of d^2 = 20: {chi2.sf(20, 2):.6f} (= e^-10); "
      f"99% ellipse d_M = {np.sqrt(chi2.ppf(0.99, 2)):.2f}")  # fmt: skip

Under the Gaussian model $d_M^2$ follows a $\chi^2$ distribution with $p=2$ degrees of freedom, so the tail
probability is a calibrated *p-value*: B is a 1-in-22,000 event, A a 1-in-8 event. The z-score ranks them the other
way round: it flags A (|z| = 2) and calls B (|z| = 1) perfectly normal.

In [ ]:
g = np.linspace(-4, 4, 200)
G = np.stack(np.meshgrid(g, g), -1).reshape(-1, 2)
dm = np.sqrt(np.einsum("ij,jk,ik->i", G, Sinv, G)).reshape(200, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for a, (Z, title) in zip(ax, [(np.abs(G).max(1).reshape(200, 200), "max |z| = 2, 3"), (dm, "Mahalanobis = 2, 3")]):
    a.scatter(*X.T, s=6, alpha=0.4)
    a.contour(g, g, Z, levels=[2, 3], colors=["tab:orange", "tab:red"])
    for n, p in [("A", A), ("B", B)]:
        a.plot(*p, "k*", ms=12)
        a.annotate(n, p + 0.15)
    a.set_title(title)
    a.set_xlabel("$x_1$ (log bytes sent, std.)")
ax[0].set_ylabel("$x_2$ (log bytes received, std.)")
plt.tight_layout()
plt.show()

## B. Thresholds and the base rate

A detector needs a **threshold** $t$: alarm when $s(\mathbf{x})>t$. Without labelled attacks we set it on normal
data only, as the $(1-\alpha)$ quantile of the scores, which gives a **false-positive rate** of about $\alpha$.
For the Gaussian model there is a closed form, $t=\chi^2_{2,1-\alpha}$.

In [ ]:
for alpha in [0.05, 0.01, 0.001]:
    s = np.einsum("ij,jk,ik->i", X, Sinv, X)
    print(
        f"alpha={alpha:<6} chi2 quantile={chi2.ppf(1 - alpha, 2):6.2f}  empirical quantile of 500 normals={np.quantile(s, 1 - alpha):6.2f}"
    )

**Base rate.** Even a good detector drowns in false alarms when attacks are rare (Axelsson 2000). Bayes' rule gives
the share of alarms that are real, $P(\text{attack}\mid\text{alarm})=\frac{\text{TPR}\,b}{\text{TPR}\,b+\text{FPR}\,(1-b)}$,
with $b$ the share of attacks among events.

In [ ]:
rows = []
for b in [0.1, 0.01, 0.001, 0.0001]:
    rows.append(
        {"base rate b": b, **{f"FPR={f}": round(an.precision_at_base_rate(0.95, f, b), 4) for f in [0.01, 0.001]}}
    )
print("precision P(attack | alarm) for TPR = 0.95")
print(pl.DataFrame(rows))
print("alarms per day at 1,000,000 events, FPR 0.01:", int(0.01 * 1_000_000))

## C. Contamination: the estimate is part of the attack surface

The mean and covariance are *estimated* from data that may already contain attacks. Add 5% of points around B to
the training data: the covariance stretches towards B and its distance shrinks. The **Minimum Covariance
Determinant** (MCD, Rousseeuw 1984) fits the covariance on the $h$ most concentrated points only, so a minority of
outliers cannot move it.

In [ ]:
rng = np.random.default_rng(1)
rows = []
for frac in [0.0, 0.05, 0.10, 0.20]:
    k = int(frac * len(X))
    Xc = np.vstack([X, B * 1.5 + rng.normal(0, 0.2, (k, 2))])
    for det in [an.Mahalanobis(), an.Mahalanobis(robust=True)]:
        det.fit(Xc)
        rows.append({"contamination": frac, "estimator": det.name, "d^2(B)": round(float(det.score(B[None])[0]), 2),
                     "d^2(A)": round(float(det.score(A[None])[0]), 2)})  # fmt: skip
print(pl.DataFrame(rows))

Ledoit-Wolf (the plain estimator here) is pulled towards B as soon as it is in the data; MCD keeps $d^2(B)$ high
until the contamination approaches its breakdown point (here half of the data minus the dimension).

## D. Distance, density and isolation

Three non-Gaussian views of the same question:
* **kNN distance:** the distance to the $k$-th nearest normal point (large = isolated).
* **LOF** (Breunig et al. 2000): the local density of a point divided by that of its neighbours (about 1 = normal).
* **Isolation Forest** (Liu et al. 2008): random axis-aligned cuts; a point that is isolated after few cuts is
  anomalous. Score $s(\mathbf{x})=2^{-E[h(\mathbf{x})]/c(n)}$ with $h$ the path length and
  $c(n)=2H(n-1)-2(n-1)/n$ the average path of an unsuccessful search in a binary tree ($H(i)\approx\ln i+0.5772$).

In [ ]:
def c(n: int) -> float:
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n


print(f"Euler's constant {np.euler_gamma:.4f}; c(256) = {c(256):.3f}: a point isolated after 4 cuts scores 2^(-4/c) = {2 ** (-4 / c(256)):.3f};"
      f" after c(256) cuts it scores 0.5")  # fmt: skip
rows = []
for det in [an.KNNDist(k=5), an.LOF(k=20), an.IForest(), an.Mahalanobis()]:
    det.fit(X)
    s = det.score(np.vstack([X, A, B]))
    rank = lambda v: int((s[:-2] >= v).sum())  # noqa: E731  (how many normal points score at least as high)
    rows.append({"detector": det.name, "normals above A": rank(s[-2]), "normals above B": rank(s[-1])})
print("rank of A and B among the 500 normal points (0 = more anomalous than every normal point)")
print(pl.DataFrame(rows))

B lies in an *empty* region between normal points, so the methods that look at local distances or densities
(kNN, LOF) and the covariance (Mahalanobis) put it above every or almost every normal point. The Isolation Forest
cuts along the axes only: B has ordinary values on each axis and needs more cuts than the corner points of the
cloud (31 normals rank higher). Axis-aligned cuts are blind to correlations; the *extended* Isolation Forest
(Hariri et al. 2021) uses random oblique cuts for this reason.

**LOF and dense attacks.** A flood (thousands of identical connections) forms a *dense* cluster: its points have
neighbours as dense as themselves, so their LOF is about 1. Notebook 02 shows that LOF misses most of the DoS rows
of KDD99 for this reason.

In [ ]:
flood = np.array([3.0, -0.5]) + rng.normal(0, 0.02, (60, 2))
lof = an.LOF(k=20).fit(X)
knn = an.KNNDist(k=5).fit(X)
print(f"flood of 60 near-identical points: mean LOF score {lof.score(flood).mean():.2f} (normal median "
      f"{np.median(lof.score(X)):.2f}), kNN distance {knn.score(flood).mean():.2f} (normal median {np.median(knn.score(X)):.2f})")  # fmt: skip
lof_in = an.LOF(k=20).fit(np.vstack([X, flood]))
print(f"if the flood is in the training data (unsupervised setting): LOF of the flood {lof_in.score(flood).mean():.2f}")

## E. The toy illustrations of the slides

The slides draw kNN radii, a LOF pair, k-means and a Gaussian mixture on small toy data. Each call below draws its
figure, stores it for the slides and prints the numbers shown on them.

In [ ]:
import plots_methods as pm

for fn in (pm.knn, pm.lof, pm.kmeans_gmm):
    fn()